# SmartTraining AI — Modèle de prédiction du risque pédagogique

## Objectif

Ce notebook présente la construction du modèle de prédiction du risque pédagogique intégré à SmartTraining.

La chaîne étudiée est :

**Dataset → Préparation des variables → Entraînement → Comparaison de modèles → Évaluation → Sauvegarde du pipeline → Utilisation par FastAPI**

### Principes méthodologiques

- Le problème est formulé comme une **classification supervisée binaire** :
  - `at_risk = 0` : apprenant non identifié comme à risque ;
  - `at_risk = 1` : apprenant identifié comme à risque.
- Le dataset utilisé pour le prototype est **synthétique**.
- La cible `at_risk` est une **weak label**.
- Trois modèles sont comparés : **Logistic Regression**, **Decision Tree** et **KNN**.
- La **Logistic Regression** est retenue comme modèle principal pour son compromis entre performance et interprétabilité.
- Les identifiants `learnerId` et `trainingId` sont exclus des variables prédictives.
- Les métriques obtenues décrivent les performances sur le dataset du prototype et ne constituent pas une validation sur une population réelle.

### Définition — *weak label*

Une **weak label** est une étiquette construite à partir de règles ou d’heuristiques, plutôt qu’à partir d’une vérité terrain observée manuellement sur des cas réels.

Dans SmartTraining, la cible `at_risk` est dérivée d’indicateurs pédagogiques tels que la progression, les scores, l’inactivité et la complétion.

Cette approche est adaptée à un prototype lorsque l’on ne dispose pas encore d’un historique réel suffisamment riche. Une version destinée à un usage opérationnel devrait être réentraînée et validée sur des données réelles anonymisées et gouvernées.


## 1. Imports et reproductibilité

Cette cellule charge les bibliothèques utilisées pour :
- manipuler les données (`pandas`, `numpy`) ;
- construire les modèles (`scikit-learn`) ;
- tracer les graphiques (`matplotlib`) ;
- sauvegarder le pipeline (`joblib`).

`RANDOM_SEED = 42` permet d'obtenir des découpages reproductibles.


In [ ]:
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay,
)

RANDOM_SEED = 42
print("Environnement prêt.")


## 2. Chargement portable du dataset

Cette cellule est volontairement compatible avec **Google Colab**.

- Dans Colab : si le CSV n'est pas déjà présent, une fenêtre permet de l'uploader.
- Hors Colab : le notebook cherche le CSV dans le dossier courant puis dans l'arborescence historique du projet.

### Interprétation 
« Le notebook original utilisait des chemins relatifs vers l'arborescence locale du projet. Cette version de démonstration rend le chargement portable pour Colab sans changer la méthodologie ML. »


In [ ]:
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    from google.colab import files

    expected = Path("/content/smarttraining_ai_dataset.csv")
    if expected.exists():
        DATASET_PATH = expected
        print("Dataset déjà présent :", DATASET_PATH)
    else:
        print("Sélectionnez le fichier smarttraining_ai_dataset.csv")
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("Aucun fichier n'a été uploadé.")
        uploaded_name = next(iter(uploaded))
        DATASET_PATH = Path("/content") / uploaded_name
else:
    candidates = [
        Path.cwd() / "smarttraining_ai_dataset.csv",
        Path("../datasets/smarttraining_ai_dataset.csv"),
        Path("datasets/smarttraining_ai_dataset.csv"),
    ]
    DATASET_PATH = next((p for p in candidates if p.exists()), None)
    if DATASET_PATH is None:
        raise FileNotFoundError(
            "Dataset introuvable. Placez smarttraining_ai_dataset.csv à côté du notebook."
        )

df = pd.read_csv(DATASET_PATH)
print("Dataset utilisé :", DATASET_PATH)
print("Dimensions :", df.shape)
df.head()


## 3. Audit rapide du dataset

Avant d’entraîner un modèle, on vérifie la qualité minimale des données :

- nombre de lignes et de colonnes ;
- répartition de la cible `at_risk` ;
- présence éventuelle de valeurs manquantes ;
- présence éventuelle de colonnes qui donneraient directement la réponse au modèle.

### Pourquoi vérifier la répartition de la cible ?

Si presque tous les apprenants appartenaient à la même classe, une bonne **accuracy** pourrait être trompeuse.  
On vérifie donc la proportion de `0` et de `1`.

### Définition rapide — *target leakage*

Une **fuite de cible** (*target leakage*) apparaît lorsqu’une variable utilisée pour l’entraînement contient directement ou indirectement l’information que le modèle doit prédire.

Exemple : utiliser une colonne `riskLevel` pour prédire `at_risk` donnerait au modèle la réponse presque directement.

> **Interprétation :** « J’ai vérifié l’absence de variables qui révéleraient directement la cible afin d’éviter une évaluation artificiellement élevée. »


In [ ]:
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))

target_counts = df["at_risk"].value_counts().sort_index()
target_percentages = (
    df["at_risk"].value_counts(normalize=True).sort_index() * 100
).round(2)

print("\nRépartition at_risk :")
print(target_counts)
print("\nPourcentage :")
print(target_percentages)

print("\nValeurs manquantes :", int(df.isna().sum().sum()))

forbidden_columns = [
    "riskScore",
    "riskLevel",
    "riskFactors",
    "recommendations",
    "status",
    "atRiskTrainings",
]
present_forbidden = [c for c in forbidden_columns if c in df.columns]
print("Colonnes à fuite de cible présentes :", present_forbidden)
assert not present_forbidden, "Fuite de cible détectée."

target_counts.plot(kind="bar")
plt.title("Répartition de la cible at_risk")
plt.xlabel("Classe (0 = non à risque, 1 = à risque)")
plt.ylabel("Nombre d'observations")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Sélection des variables (*features*)

Une **feature** est une variable utilisée par le modèle pour effectuer sa prédiction.

Ici, les variables décrivent notamment :

- **progression** : pourcentage d’avancement, leçons terminées ;
- **résultats** : score moyen, ratio de score ;
- **quiz** : nombre de quiz terminés et taux de complétion ;
- **activité** : nombre d’événements et activité moyenne ;
- **engagement temporel** : nombre de jours depuis la dernière activité ;
- **historique** : formations commencées et terminées.

`learnerId` et `trainingId` sont volontairement exclus.

### Pourquoi exclure les identifiants ?

Un identifiant technique n’a pas de signification pédagogique.  
Par exemple, `learnerId = 42` ne signifie pas que l’apprenant est plus à risque que `learnerId = 15`.

> **Interprétation :** « Les identifiants servent à rattacher une prédiction à un utilisateur et à une formation, mais ils ne participent pas au calcul de la prédiction. »


In [ ]:
target_column = "at_risk"
identifier_columns = ["learnerId", "trainingId"]

feature_columns = [
    "progressPercentage",
    "averageScore",
    "completedLessons",
    "totalLessons",
    "completedQuizzes",
    "totalQuizzes",
    "totalEvents",
    "totalTrainingsStarted",
    "totalTrainingsCompleted",
    "lessonCompletionRate",
    "quizCompletionRate",
    "scoreRatio",
    "daysSinceLastActivity",
    "avgEventsPerTraining",
]

assert not set(identifier_columns).intersection(feature_columns)

X = df[feature_columns]
y = df[target_column]

print("Nombre de features :", len(feature_columns))
print("Features utilisées :")
for i, feature in enumerate(feature_columns, 1):
    print(f"{i:02d}. {feature}")

print("\nX :", X.shape, "| y :", y.shape)


## 5. Découpage entraînement / test

Le dataset est séparé en deux parties :

- **80 % entraînement** : données utilisées pour apprendre les paramètres du modèle ;
- **20 % test** : données gardées de côté pour l’évaluation finale.

Le paramètre `stratify=y` conserve approximativement la même proportion de classes `0/1` dans les deux ensembles.

### Pourquoi garder un jeu de test séparé ?

Si on évaluait le modèle uniquement sur les données qu’il a déjà vues pendant l’entraînement, on mesurerait surtout sa capacité à mémoriser ces données.

Le jeu de test permet d’évaluer sa capacité à **généraliser** à des observations non vues pendant l’apprentissage.

> **Interprétation :** « Le jeu de test est isolé de l’entraînement afin d’obtenir une évaluation plus réaliste du comportement du modèle sur des données non vues. »


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y,
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("\nRépartition train (%) :")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))
print("\nRépartition test (%) :")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))


## 6. Trois modèles comparés

### Logistic Regression
Modèle de **classification binaire** qui estime notamment une probabilité de la classe positive.  
Il est apprécié ici pour sa simplicité et son interprétabilité.

### Decision Tree
Construit une succession de règles de décision.  
Il est intuitif, mais peut devenir trop spécifique aux données d’entraînement s’il est trop profond.

### KNN
Classe un nouvel exemple en regardant les exemples les plus proches dans l’espace des variables.

---

### Pourquoi utiliser un `Pipeline` scikit-learn ?

Un `Pipeline` regroupe dans un seul objet :

1. le traitement des valeurs manquantes ;
2. la standardisation éventuelle ;
3. le modèle.

Cela garantit que **les mêmes transformations** sont appliquées pendant l’entraînement et lors des prédictions futures.

### Rôle de `StandardScaler`

Certaines variables n’ont pas la même échelle : par exemple un pourcentage peut aller jusqu’à `100`, alors qu’un ratio peut rester entre `0` et `1`.

Le `StandardScaler` remet les variables sur des échelles comparables, ce qui est utile pour Logistic Regression et KNN.

> **Interprétation :** « Le pipeline évite de dissocier le prétraitement du modèle et permet de sauvegarder toute la chaîne d’inférence dans un seul fichier. »


In [ ]:
logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
])

decision_tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(random_state=RANDOM_SEED)),
])

knn_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier()),
])

models = {
    "Logistic Regression": logistic_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "KNN": knn_pipeline,
}

print("Modèles comparés :", list(models.keys()))


## 7. Validation croisée — 5 plis

La validation croisée évite de juger un modèle sur un seul découpage du jeu d’entraînement.

Avec **5 plis**, le jeu d’entraînement est divisé en cinq parties.  
À chaque tour :

- quatre parties servent à entraîner ;
- la cinquième sert à valider ;
- le rôle des parties change jusqu’à ce que les cinq aient servi de validation.

`StratifiedKFold` conserve la proportion des classes dans chaque pli.

### Pourquoi utiliser le F1-score ?

Le **F1-score** combine :

- **Precision** : parmi les apprenants prédits à risque, combien appartiennent réellement à la classe `at_risk = 1` dans le dataset ?
- **Recall** : parmi les apprenants de classe `at_risk = 1`, combien le modèle détecte-t-il ?
- **F1-score** : compromis entre Precision et Recall.

> **Interprétation :** « J’ai utilisé la validation croisée pour éviter de sélectionner le modèle sur une seule partition des données, et le F1-score car la classe à risque est particulièrement importante à détecter. »


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)

cv_results = {}

for model_name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1",
    )
    cv_results[model_name] = scores
    print(model_name)
    print("  F1 par pli :", scores.round(3))
    print("  F1 moyen   :", round(scores.mean(), 3))
    print("  Écart-type :", round(scores.std(), 3))
    print("-" * 55)


## 8. Comparaison sur le jeu de test

Après l’entraînement, chaque modèle est évalué sur le jeu de test avec plusieurs métriques.

### Lecture rapide des métriques

- **Accuracy** : proportion totale de prédictions correctes.
- **Precision** : fiabilité des prédictions positives.
- **Recall** : capacité à retrouver les cas positifs.
- **F1-score** : équilibre entre Precision et Recall.
- **ROC AUC** : capacité globale du modèle à distinguer les deux classes à travers différents seuils.

> **Limite méthodologique :** ces métriques concernent le dataset synthétique et la weak label du prototype. Elles démontrent le fonctionnement de la chaîne ML, pas une efficacité validée sur une population réelle.


In [ ]:
test_results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    report = classification_report(
        y_test,
        y_pred,
        output_dict=True,
        zero_division=0,
    )

    test_results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_at_risk": report["1"]["precision"],
        "recall_at_risk": report["1"]["recall"],
        "f1_at_risk": report["1"]["f1-score"],
        "roc_auc": roc_auc_score(y_test, y_pred_proba),
    })

results_df = (
    pd.DataFrame(test_results)
    .sort_values("f1_at_risk", ascending=False)
    .reset_index(drop=True)
)

results_df.round(3)


## 9. Optimisation de la Logistic Regression

Après avoir comparé plusieurs familles de modèles, on optimise la Logistic Regression avec `GridSearchCV`.

### Qu’est-ce qu’un hyperparamètre ?

Un **hyperparamètre** est un réglage choisi avant ou pendant l’entraînement, contrairement aux coefficients internes appris automatiquement par le modèle.

Ici, on teste notamment :

- `C` : contrôle la régularisation ;
- `class_weight` : permet éventuellement de donner davantage d’importance à une classe.

`GridSearchCV` teste plusieurs combinaisons et conserve celle qui obtient le meilleur F1-score en validation croisée.

> **Interprétation :** « Je n’ai pas choisi les paramètres arbitrairement : plusieurs combinaisons sont évaluées avec validation croisée. »


In [ ]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"],
}

grid_search = GridSearchCV(
    logistic_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

print("Meilleurs paramètres :", grid_search.best_params_)
print("Meilleur F1 CV       :", round(grid_search.best_score_, 3))


## 10. Évaluation finale du modèle retenu

### Matrice de confusion

La matrice de confusion montre quatre situations :

- **vrai négatif** : non-risque correctement identifié ;
- **faux positif** : risque signalé alors que la cible vaut 0 ;
- **faux négatif** : risque non détecté alors que la cible vaut 1 ;
- **vrai positif** : risque correctement détecté.

Dans un contexte pédagogique, les **faux négatifs** sont particulièrement intéressants à surveiller : ils représentent des apprenants à risque que le modèle n’aurait pas signalés.

### Courbe ROC et ROC AUC

La courbe ROC analyse le compromis entre détection des positifs et faux positifs pour plusieurs seuils.

Le **ROC AUC** résume cette capacité de discrimination en une valeur.

> **Interprétation :** « Je ne regarde pas seulement l’accuracy ; j’analyse aussi les erreurs de classification et la capacité de discrimination du modèle. »


In [ ]:
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

report_best = classification_report(
    y_test,
    y_pred_best,
    output_dict=True,
    zero_division=0,
)

print(classification_report(y_test, y_pred_best, zero_division=0))

cm = confusion_matrix(y_test, y_pred_best)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non risque", "Risque"]).plot()
plt.title("Matrice de confusion — Logistic Regression")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_pred_proba_best)
plt.title("Courbe ROC — Logistic Regression")
plt.tight_layout()
plt.show()

print("ROC AUC :", round(roc_auc_score(y_test, y_pred_proba_best), 3))


## 11. Démonstration d’une prédiction

Cette cellule illustre une prédiction complète sur un profil d’apprenant exemple.

Le profil choisi représente volontairement un apprenant avec :
- faible progression ;
- score limité ;
- peu d’activité ;
- longue période d’inactivité.

Le modèle retourne deux informations différentes :

- `predict()` → la **classe finale** (`0` ou `1`) ;
- `predict_proba()` → la **probabilité associée à la classe à risque**.

### Important

Une probabilité n’est pas une certitude.  
Elle représente la sortie du modèle sur la base des données et de l’apprentissage réalisé.

> **Interprétation :** « La prédiction est une aide à la décision. Elle ne remplace pas le formateur et ne doit pas être interprétée comme une vérité absolue. »


In [ ]:
sample_learner = pd.DataFrame([{
    "progressPercentage": 25,
    "averageScore": 42,
    "completedLessons": 1,
    "totalLessons": 8,
    "completedQuizzes": 0,
    "totalQuizzes": 3,
    "totalEvents": 2,
    "totalTrainingsStarted": 1,
    "totalTrainingsCompleted": 0,
    "lessonCompletionRate": 0.125,
    "quizCompletionRate": 0.0,
    "scoreRatio": 0.42,
    "daysSinceLastActivity": 28,
    "avgEventsPerTraining": 2.0,
}])

prediction = int(best_model.predict(sample_learner)[0])
probability = float(best_model.predict_proba(sample_learner)[0][1])

print("Prédiction at_risk :", prediction)
print("Probabilité risque :", round(probability, 3))
print(
    "Interprétation     :",
    "apprenant prédit à risque" if prediction else "apprenant non prédit à risque",
)


## 12. Interprétabilité — coefficients de la Logistic Regression

La Logistic Regression fournit un coefficient pour chaque feature.

Dans l’espace standardisé :

- un coefficient **positif** tend à augmenter la probabilité de `at_risk = 1` ;
- un coefficient **négatif** tend à la diminuer ;
- une valeur absolue plus élevée indique une influence plus importante dans le modèle.

### Prudence d’interprétation

Un coefficient montre une relation utilisée par le modèle, mais **ne prouve pas une causalité pédagogique**.

Par exemple, une forte association entre inactivité et risque ne signifie pas automatiquement que l’inactivité est l’unique cause du risque.

> **Interprétation :** « L’un des avantages de la Logistic Regression est que ses coefficients permettent une lecture plus transparente qu’un modèle beaucoup plus complexe. »


In [ ]:
logistic_model = best_model.named_steps["model"]

coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": logistic_model.coef_[0],
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

coefficients.sort_values(
    "abs_coefficient",
    ascending=False,
).reset_index(drop=True)


## 13. Sauvegarde du pipeline et des métadonnées

Une fois le modèle entraîné, il n’est pas réentraîné à chaque requête.

### Fichier `.joblib`

Le fichier `risk_prediction_pipeline.joblib` contient le pipeline entraîné :

- imputation ;
- standardisation ;
- Logistic Regression ;
- paramètres appris.

FastAPI recharge ce fichier en mémoire au démarrage grâce à `joblib.load()`.

### Fichier `model_metadata.json`

Ce fichier joue le rôle de **carte d’identité du modèle** :

- nom et version ;
- liste des features ;
- méthode d’entraînement ;
- paramètres retenus ;
- métriques ;
- information sur la `weak_label`.

### Pourquoi sauvegarder les deux ?

Le `.joblib` permet d’exécuter les prédictions.  
Le JSON permet d’expliquer, documenter et versionner le modèle.

> **Interprétation :** « J’ai séparé l’artefact exécutable du modèle de ses métadonnées descriptives. »


In [ ]:
if IN_COLAB:
    MODELS_DIR = Path("/content/models")
else:
    MODELS_DIR = Path.cwd() / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "risk_prediction_pipeline.joblib"
METADATA_PATH = MODELS_DIR / "model_metadata.json"

joblib.dump(best_model, MODEL_PATH)

metadata = {
    "project": "SmartTraining AI",
    "model_version": "1.0.0",
    "model_name": "LogisticRegression",
    "problem_type": "binary_classification",
    "target": "at_risk",
    "target_description": {
        "0": "apprenant non à risque",
        "1": "apprenant à risque",
    },
    "label_type": "weak_label",
    "label_warning": (
        "La cible at_risk est construite par règles pour le prototype PFE. "
        "Un modèle de production devra être réentraîné avec des résultats réels observés."
    ),
    "features": feature_columns,
    "identifier_columns_excluded": identifier_columns,
    "excluded_columns_to_avoid_target_leakage": [
        "riskScore",
        "riskLevel",
        "riskFactors",
        "recommendations",
        "status",
        "atRiskTrainings",
    ],
    "training_method": {
        "train_test_split": "80/20",
        "random_seed": RANDOM_SEED,
        "cross_validation": "StratifiedKFold 5 plis",
        "selection_metric": "F1",
        "models_compared": [
            "Logistic Regression",
            "Decision Tree",
            "KNN",
        ],
    },
    "best_params": grid_search.best_params_,
    "best_cv_f1": float(grid_search.best_score_),
    "test_metrics": {
        "accuracy": float(accuracy_score(y_test, y_pred_best)),
        "precision_at_risk": float(report_best["1"]["precision"]),
        "recall_at_risk": float(report_best["1"]["recall"]),
        "f1_at_risk": float(report_best["1"]["f1-score"]),
        "roc_auc": float(roc_auc_score(y_test, y_pred_proba_best)),
    },
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}

with METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Modèle sauvegardé    :", MODEL_PATH)
print("Métadonnées sauvegardées :", METADATA_PATH)


## 14. Option Colab : télécharger les artefacts

À utiliser seulement si vous souhaitez récupérer les fichiers générés depuis Colab.

Mettez `DOWNLOAD_ARTIFACTS = True`, puis exécutez la cellule.


In [ ]:
DOWNLOAD_ARTIFACTS = False

if DOWNLOAD_ARTIFACTS:
    if IN_COLAB:
        from google.colab import files
        files.download(str(MODEL_PATH))
        files.download(str(METADATA_PATH))
    else:
        print("Hors Colab : les fichiers sont déjà dans", MODELS_DIR)
else:
    print("Téléchargement automatique désactivé.")


# Conclusion

Le processus Machine Learning de SmartTraining suit les étapes suivantes :

**Dataset synthétique**  
↓  
**Construction de la cible `at_risk` par weak label**  
↓  
**Sélection des variables explicatives**  
↓  
**Découpage Train/Test 80/20**  
↓  
**Comparaison Logistic Regression / Decision Tree / KNN**  
↓  
**Validation croisée stratifiée à 5 plis**  
↓  
**Optimisation de la Logistic Regression par GridSearchCV**  
↓  
**Évaluation : Precision, Recall, F1-score, ROC AUC et matrice de confusion**  
↓  
**Sauvegarde du pipeline dans un fichier `.joblib`**  
↓  
**Chargement du pipeline par FastAPI pour l’inférence**

## Limites et perspectives

Le dataset utilisé dans ce prototype est **synthétique** et la variable cible `at_risk` est une **weak label**, c’est-à-dire une étiquette construite à partir de règles et d’heuristiques plutôt qu’à partir d’une vérité terrain observée sur une population réelle.

Cette approche permet de valider la **chaîne technique complète** de Machine Learning et son intégration dans le LMS.

Une évolution vers un usage industriel nécessiterait notamment :

- des données réelles anonymisées et gouvernées ;
- une définition de la cible fondée sur des résultats pédagogiques observés ;
- un réentraînement sur ces données ;
- une nouvelle validation statistique ;
- un suivi des performances du modèle dans le temps.

> **Synthèse :** le prototype démontre une architecture ML complète, reproductible et intégrée à SmartTraining, tout en distinguant clairement la validation technique de la validation métier sur données réelles.
